In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('heart_disease_uci.csv')
print(df[['age', 'trestbps', 'chol', 'thalch', 'oldpeak']].describe())

              age    trestbps        chol      thalch     oldpeak
count  920.000000  861.000000  890.000000  865.000000  858.000000
mean    53.510870  132.132404  199.130337  137.545665    0.878788
std      9.424685   19.066070  110.780810   25.926276    1.091226
min     28.000000    0.000000    0.000000   60.000000   -2.600000
25%     47.000000  120.000000  175.000000  120.000000    0.000000
50%     54.000000  130.000000  223.000000  140.000000    0.500000
75%     60.000000  140.000000  268.000000  157.000000    1.500000
max     77.000000  200.000000  603.000000  202.000000    6.200000


In [6]:
# Clinical plausibility ranges for continuous variables
# These are NOT normal ranges — they define physiologically possible values
# Any value outside these ranges is considered a data capture error, not a clinical outlier
#
# Sources:
# - age: standard adult range (AHA, WHO)
# - trestbps: below 60 = hemodynamic shock; above 250 = extreme hypertensive emergency
#   Reference: JNC 8 Guidelines, AHA Blood Pressure Categories
# - chol: below 100 = rare genetic condition (abetalipoproteinemia);
#   above 600 = extreme familial hypercholesterolemia
#   Reference: AHA Cholesterol Guidelines, Circulation 2018
# - thalch: below 50 = severe bradycardia/non-exercise state;
#   above 220 = exceeds theoretical maximum (220 - age) even for youngest patient (28)
#   Reference: ACSM Guidelines for Exercise Testing, 10th ed.
# - oldpeak: below -2 = measurement artifact; above 7 = exceeds documented clinical range
#   Reference: Detrano et al. (1989), AHA Exercise Standards

clinical_ranges = {
    'age':      (18, 100),
    'trestbps': (60, 250),
    'chol':     (100, 600),
    'thalch':   (50, 220),
    'oldpeak':  (-2, 7)
}

for col, (low, high) in clinical_ranges.items():
    n_out = ((df[col] < low) | (df[col] > high)).sum()
    n_total = df[col].notna().sum()
    print(f"{col}: {n_out} values outside [{low}, {high}] out of {n_total}")

age: 0 values outside [18, 100] out of 920
trestbps: 1 values outside [60, 250] out of 861
chol: 174 values outside [100, 600] out of 890
thalch: 0 values outside [50, 220] out of 865
oldpeak: 1 values outside [-2, 7] out of 858


In [9]:
# Ver el valor exacto que está fuera de rango en trestbps
print(df[df['trestbps'] < 60][['age', 'sex', 'trestbps']])

# Ver el valor exacto en oldpeak  
print(df[df['oldpeak'] < -2][['age', 'sex', 'oldpeak']])

# chol breakdown
print(f"Chol zeros: {(df['chol'] == 0).sum()}")
print(f"Chol < 100 (non-zero): {((df['chol'] < 100) & (df['chol'] > 0)).sum()}")
print(f"Chol > 600: {(df['chol'] > 600).sum()}")

     age   sex  trestbps
753   55  Male       0.0
     age   sex  oldpeak
615   46  Male     -2.6
Chol zeros: 172
Chol < 100 (non-zero): 1
Chol > 600: 1
